# NB82 — The Solenoid Metric

**From Configuration Space to the (t, R) Line Element**

The solenoid Lagrangian (NB43) defines a natural Riemannian metric on
configuration space through the kinetic energy $T = \frac{1}{2}\dot\theta^\top W \dot\theta$.
NB46 established that the algebraic sector (Cayley graph) is Ricci-flat,
and all curvature resides in the geometric sector (radial nesting).

This notebook takes the next step: we derive the **metric on the solenoid leaf**,
transform to the physically natural **(t, R) coordinates** (time + covering residuals),
and extract structural identities connecting the metric invariants to known
solenoid quantities.

**New identities**: #181–#186 (6 new, running total 177).

**Contents**:
1. Configuration space metric $W = \mathrm{diag}(P_0, \ldots, P_4)$
2. Zero-mode inertia and the 179 connection
3. The $(t, R)$ coordinate transformation — Jacobian
4. The full $(t, R)$ metric and its structure
5. Metric determinant and volume form
6. Stiffness in $(t, R)$ coordinates — time as zero mode
7. The cascade conformal factor
8. Scorecard

In [1]:
import sys, os
import numpy as np
from fractions import Fraction
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import SA, RHO, OMEGA

primes = SA.primes          # [2, 3, 5, 7]
n_levels = len(primes) + 1  # 5 (theta_0 through theta_4)
P4 = SA.P                   # 210

# Primorials P_k = prod(primes[:k])
PRIMORIALS = [1]
for p in primes:
    PRIMORIALS.append(PRIMORIALS[-1] * p)
# PRIMORIALS = [1, 2, 6, 30, 210]
print("Primorials P_k:", PRIMORIALS)
print(f"Primes: {primes}")
print(f"P_4 = {P4}, rho = 1/sqrt({P4}) = {RHO:.6f}")
print(f"omega = 2*pi = {OMEGA:.6f}")

Primorials P_k: [1, 2, 6, 30, 210]
Primes: [2, 3, 5, 7]
P_4 = 210, rho = 1/sqrt(210) = 0.069007
omega = 2*pi = 6.283185


## §1 — The Configuration Space Metric

The solenoid Lagrangian (NB43) is:

$$L = T - V = \frac{1}{2}\dot\theta^\top W \dot\theta
    - \frac{\kappa}{2}\,\theta^\top K \theta$$

where:
- $W = \mathrm{diag}(P_0, P_1, P_2, P_3, P_4) = \mathrm{diag}(1, 2, 6, 30, 210)$ — the **mass matrix** (primorial inertia)
- $K$ — the **stiffness matrix** from covering constraints $(\theta_k - p_{k+1}\theta_{k+1})^2$

The kinetic energy $T = \frac{1}{2}g_{ij}\dot\theta^i\dot\theta^j$ defines
the **configuration space metric** $g = W$.

This is a flat metric on $T^5$ — a 5-torus with primorial radii.
All curvature comes from the potential $V$ (the stiffness $K$).

In [2]:
# ── Build W (mass matrix) and K (stiffness matrix) ──
W = np.diag([Fraction(Pk) for Pk in PRIMORIALS])
W_float = np.diag([float(Pk) for Pk in PRIMORIALS])

# Stiffness from covering constraints: V = (kappa/2) sum_i (theta_i - p_{i+1} theta_{i+1})^2
K = np.zeros((n_levels, n_levels), dtype=object)
for i in range(n_levels):
    for j in range(n_levels):
        K[i, j] = Fraction(0)

for i in range(n_levels - 1):
    p = primes[i]  # p_{i+1}
    K[i, i] += Fraction(1)
    K[i + 1, i + 1] += Fraction(p * p)
    K[i, i + 1] += Fraction(-p)
    K[i + 1, i] += Fraction(-p)

print("W = diag(P_0, ..., P_4) =", [PRIMORIALS[k] for k in range(n_levels)])
print()

print("K (stiffness matrix):")
for i in range(n_levels):
    row = [f"{int(K[i,j]):>5d}" for j in range(n_levels)]
    print("  [" + ", ".join(row) + "]")

# Verify zero mode
v0 = np.array([Fraction(1, Pk) for Pk in PRIMORIALS])
Kv0 = np.array([sum(K[i, j] * v0[j] for j in range(n_levels)) for i in range(n_levels)])
print(f"\nK * v_0 = {list(Kv0)}  (zero mode check)")

# Determinant of W
det_W = Fraction(1)
for Pk in PRIMORIALS:
    det_W *= Fraction(Pk)
print(f"\ndet(W) = prod(P_k) = {det_W}")

# Prime factorization of det(W)
from sympy import factorint
factors = factorint(int(det_W))
print(f"       = " + " × ".join(f"{p}^{e}" for p, e in sorted(factors.items())))
print(f"\nExponent of p_k in det(W):")
for k, p in enumerate(primes):
    exp = factors.get(p, 0)
    print(f"  p_{k+1} = {p}: exponent = {exp} = {n_levels} - {k+1} = n_levels - k")

W = diag(P_0, ..., P_4) = [1, 2, 6, 30, 210]

K (stiffness matrix):
  [    1,    -2,     0,     0,     0]
  [   -2,     5,    -3,     0,     0]
  [    0,    -3,    10,    -5,     0]
  [    0,     0,    -5,    26,    -7]
  [    0,     0,     0,    -7,    49]

K * v_0 = [Fraction(0, 1), Fraction(0, 1), Fraction(0, 1), Fraction(0, 1), Fraction(0, 1)]  (zero mode check)

det(W) = prod(P_k) = 75600
       = 2^4 × 3^3 × 5^2 × 7^1

Exponent of p_k in det(W):
  p_1 = 2: exponent = 4 = 5 - 1 = n_levels - k
  p_2 = 3: exponent = 3 = 5 - 2 = n_levels - k
  p_3 = 5: exponent = 2 = 5 - 3 = n_levels - k
  p_4 = 7: exponent = 1 = 5 - 4 = n_levels - k


## §2 — The Zero-Mode Inertia and the 179 Connection

The **exact solenoid leaf** is parameterized by a single angle $\varphi$:

$$\theta_k = \frac{\varphi}{P_k} + \frac{2\pi j_k}{p_k}$$

The tangent vector is $v_0 = (1/P_0, 1/P_1, \ldots, 1/P_4)$ — 
the zero mode of $K$ (NB43 identity #41), the **influx vector**.

The **induced 1D metric** on the solenoid leaf is the $W$-norm:

$$g_{\text{1D}} = \|v_0\|_W^2 = v_0^\top W\, v_0 = \sum_{k=0}^{4} P_k \cdot \frac{1}{P_k^2} = \sum_{k=0}^{4} \frac{1}{P_k}$$

This is the effective "inertia" of the influx mode.

In [3]:
# ── Zero-mode inertia ──
g_1D = sum(Fraction(1, Pk) for Pk in PRIMORIALS)
print(f"g_1D = sum(1/P_k) = {g_1D} = {float(g_1D):.10f}")
print(f"     = {g_1D.numerator}/{g_1D.denominator}")
print()

# Verify: 179 is prime
from sympy import isprime
print(f"179 is prime: {isprime(179)}")
print(f"105 = {factorint(105)} = 3 × 5 × 7 = P_4 / P_1")
print()

# ── Connection to eigenvalue product (NB43 identity #42) ──
# Eigenvalues of W^{-1}K: compute numerically, verify exact product
W_inv = np.diag([1.0 / Pk for Pk in PRIMORIALS])
K_float = np.array([[float(K[i, j]) for j in range(n_levels)] for i in range(n_levels)])
eigvals = np.sort(np.linalg.eigvalsh(W_inv @ K_float))
omega_sq = eigvals[1:]  # skip the zero eigenvalue
print(f"omega^2 eigenvalues: {omega_sq}")
print(f"  sum(omega^2) = {sum(omega_sq):.10f}  (expect 94/15 = {94/15:.10f})")
print(f"  prod(omega^2) = {np.prod(omega_sq):.10f}  (expect 179/180 = {179/180:.10f})")
print()

# ── The product bridge ──
prod_omega_sq = Fraction(179, 180)  # NB43 identity #42
lam_P4 = 12  # lambda(210) = lcm(1,2,4,6)
p4 = primes[-1]  # 7

bridge = g_1D * Fraction(p4, lam_P4)
print(f"g_1D * p_4 / lambda(P_4) = {g_1D} * {p4}/{lam_P4} = {bridge}")
print(f"Pi(omega^2) = {prod_omega_sq}")
print(f"Match: {bridge == prod_omega_sq}")
print()
print(f"Rearranged: g_1D = Pi(omega^2) * lambda(P_4) / p_4")
print(f"  = ({prod_omega_sq.numerator}/{prod_omega_sq.denominator}) * ({lam_P4}/{p4})")
check = prod_omega_sq * Fraction(lam_P4, p4)
print(f"  = {check} = {g_1D}")
print(f"  Verified: {check == g_1D}")

g_1D = sum(1/P_k) = 179/105 = 1.7047619048
     = 179/105

179 is prime: True
105 = {3: 1, 5: 1, 7: 1} = 3 × 5 × 7 = P_4 / P_1

omega^2 eigenvalues: [0.45320788 0.83713279 1.60601269 3.13879687]
  sum(omega^2) = 6.0351502303  (expect 94/15 = 6.2666666667)
  prod(omega^2) = 1.9125112011  (expect 179/180 = 0.9944444444)

g_1D * p_4 / lambda(P_4) = 179/105 * 7/12 = 179/180
Pi(omega^2) = 179/180
Match: True

Rearranged: g_1D = Pi(omega^2) * lambda(P_4) / p_4
  = (179/180) * (12/7)
  = 179/105 = 179/105
  Verified: True


In [4]:
# ── Fix: symmetrize the eigenvalue problem ──
# W^{-1}K is NOT symmetric. Use W^{-1/2} K W^{-1/2} instead.
W_sqrt_inv = np.diag([1.0 / np.sqrt(Pk) for Pk in PRIMORIALS])
S = W_sqrt_inv @ K_float @ W_sqrt_inv
eigvals_sym = np.sort(np.linalg.eigvalsh(S))
omega_sq = eigvals_sym[1:]  # skip zero eigenvalue

print("Corrected omega^2 (generalized eigenvalues of Kv = omega^2 Wv):")
print(f"  {omega_sq}")
sum_exact = Fraction(94, 15)
prod_exact = Fraction(179, 180)
print(f"  sum  = {sum(omega_sq):.10f}  (exact: {sum_exact} = {float(sum_exact):.10f})")
print(f"  prod = {np.prod(omega_sq):.10f}  (exact: {prod_exact} = {float(prod_exact):.10f})")

# Verify to high precision using exact trace and product
# tr(W^{-1}K) gives the sum of ALL eigenvalues (including zero)
tr_WinvK = sum(Fraction(K[i, i], PRIMORIALS[i]) for i in range(n_levels))
print(f"\ntr(W^{{-1}}K) = {tr_WinvK} = sum(omega^2) + 0 = {float(tr_WinvK):.10f}")
assert tr_WinvK == sum_exact, f"Trace mismatch: {tr_WinvK} != {sum_exact}"
print(f"Exact match: sum(omega^2) = {sum_exact}  ✓")

Corrected omega^2 (generalized eigenvalues of Kv = omega^2 Wv):
  [0.22062139 0.7481806  1.65280644 3.64505823]
  sum  = 6.2666666667  (exact: 94/15 = 6.2666666667)
  prod = 0.9944444444  (exact: 179/180 = 0.9944444444)

tr(W^{-1}K) = 94/15 = sum(omega^2) + 0 = 6.2666666667
Exact match: sum(omega^2) = 94/15  ✓


### Identity #181: Zero-Mode Inertia

$$g_{\text{1D}} = \|v_0\|_W^2 = \sum_{k=0}^{4} \frac{1}{P_k} = \frac{179}{105}$$

where 179 is prime and $105 = 3 \cdot 5 \cdot 7 = P_4/P_1$.

The effective "inertia" of the influx vector — the mass of the zero mode
in the $W$-metric — is an exact rational with prime numerator.

### Identity #182: Zero-Mode ↔ Eigenvalue Product Bridge

$$g_{\text{1D}} \cdot \frac{p_4}{\lambda(P_4)} = \prod_{i=1}^{4} \omega_i^2$$

Equivalently: $\frac{179}{105} \cdot \frac{7}{12} = \frac{179}{180}$.

This connects the **zero-mode geometry** (influx inertia) to the
**oscillation spectrum** (covering constraint frequencies) through
the group exponent $\lambda(210) = 12$ and the outermost prime $p_4 = 7$.

The 179 appearing in both numerators is not coincidence — it is the
signature of the primorial harmonic sum, linking the "stillness"
of the zero mode to the "vibration" of the normal modes.

## §3 — The $(t, R)$ Coordinate Transformation

The physically natural coordinates are:
- $t$ (time): $\theta_0 = \omega t$
- $R_k$ (covering residuals): $R_k = p_{k+1}\theta_{k+1} - \theta_k$, $k = 0,\ldots, 3$

The residuals $R_k$ measure departure from perfect nesting — these are the
dynamical variables of the cascade ODE (NB79–81).

Inverting the recurrence $R_k = p_{k+1}\theta_{k+1} - \theta_k$ gives:

$$\theta_k = \frac{1}{P_k}\left(\omega t + \sum_{j=0}^{k-1} P_j \,R_j\right)$$

This defines the Jacobian $M = \partial\theta / \partial(t, R)$: a $5 \times 5$
**lower-triangular** matrix.

In [5]:
# ── Build the Jacobian M = d(theta) / d(t, R_0, R_1, R_2, R_3) ──
# theta_k = (omega*t + sum_{j<k} P_j * R_j) / P_k
#
# M[k, 0] = omega / P_k           (time column)
# M[k, j+1] = P_j / P_k           for j < k  (R columns)
# M[k, j+1] = 0                   for j >= k

omega_frac = Fraction(1)  # work in units where omega = 1; restore later
# Actually, keep omega symbolic by using omega = 2*pi later
# For exact arithmetic, represent M with omega as a factor in column 0

M = np.zeros((n_levels, n_levels), dtype=object)
for i in range(n_levels):
    for j in range(n_levels):
        M[i, j] = Fraction(0)

for k in range(n_levels):
    M[k, 0] = Fraction(1, PRIMORIALS[k])  # omega/P_k (omega factored out)
    for j in range(k):
        M[k, j + 1] = Fraction(PRIMORIALS[j], PRIMORIALS[k])

print("Jacobian M (with omega factored from column 0):")
print("  M[k, 0] = 1/P_k  (× omega)")
print("  M[k, j+1] = P_j/P_k  for j < k")
print()
for k in range(n_levels):
    row = []
    for j in range(n_levels):
        f = M[k, j]
        if j == 0:
            row.append(f"ω/{PRIMORIALS[k]}" if f != Fraction(0) else "0")
        else:
            row.append(f"{f}" if f != Fraction(0) else "0")
    print(f"  [{', '.join(f'{s:>8s}' for s in row)}]")

# Determinant (lower triangular = product of diagonal)
det_M = Fraction(1)
for k in range(n_levels):
    if k == 0:
        det_M *= Fraction(1, PRIMORIALS[k])  # omega/P_0 with omega factored
    else:
        det_M *= Fraction(PRIMORIALS[k - 1], PRIMORIALS[k])
# det_M has omega factored out of column 0

print(f"\nDiagonal entries:")
for k in range(n_levels):
    if k == 0:
        print(f"  M[{k},{k}] = omega/P_{k} = omega/{PRIMORIALS[k]}")
    else:
        print(f"  M[{k},{k}] = P_{k-1}/P_{k} = {PRIMORIALS[k-1]}/{PRIMORIALS[k]} = 1/p_{k} = 1/{primes[k-1]}")

diag_prod = Fraction(1)
for k in range(1, n_levels):
    diag_prod *= Fraction(1, primes[k - 1])
print(f"\nProduct of R-diagonal: prod(1/p_k) = {diag_prod} = 1/P_4")
print(f"det(M) = omega * {diag_prod} = omega/P_4 = omega/{P4}")

Jacobian M (with omega factored from column 0):
  M[k, 0] = 1/P_k  (× omega)
  M[k, j+1] = P_j/P_k  for j < k

  [     ω/1,        0,        0,        0,        0]
  [     ω/2,      1/2,        0,        0,        0]
  [     ω/6,      1/6,      1/3,        0,        0]
  [    ω/30,     1/30,     1/15,      1/5,        0]
  [   ω/210,    1/210,    1/105,     1/35,      1/7]

Diagonal entries:
  M[0,0] = omega/P_0 = omega/1
  M[1,1] = P_0/P_1 = 1/2 = 1/p_1 = 1/2
  M[2,2] = P_1/P_2 = 2/6 = 1/p_2 = 1/3
  M[3,3] = P_2/P_3 = 6/30 = 1/p_3 = 1/5
  M[4,4] = P_3/P_4 = 30/210 = 1/p_4 = 1/7

Product of R-diagonal: prod(1/p_k) = 1/210 = 1/P_4
det(M) = omega * 1/210 = omega/P_4 = omega/210


### Identity #183: Jacobian Determinant

$$\det(M) = \frac{\omega}{P_4}$$

The coordinate transformation from $\theta$-space to $(t, R)$-space
contracts volumes by exactly $\omega / P_4 = 2\pi / 210$.

The $R$-diagonal is $\prod_{k=1}^{4} 1/p_k = 1/P_4$: each prime orbit
contributes a $1/p_k$ contraction factor.

## §4 — The Full $(t, R)$ Metric

The metric in $(t, R)$-coordinates is $g_{(t,R)} = M^\top W M$.
Since $W = \mathrm{diag}(P_k)$ and $M$ is lower-triangular, the
resulting metric has a **tail-sum factored** structure:

$$g_{ab} = w_a \cdot w_b \cdot s_{\max(a',b')+1}$$

where $w_0 = \omega$, $w_{j+1} = P_j$, and $s_m = \sum_{k=m}^{4} 1/P_k$
is the primorial tail sum starting at level $m$.

In [6]:
# ── Compute g_{(t,R)} = M^T W M ──
# Note: omega is factored out of column 0 of M. We'll track it symbolically.

# First compute M^T W M in exact arithmetic (with omega=1 in M, restore later)
g_tR = np.zeros((n_levels, n_levels), dtype=object)
for a in range(n_levels):
    for b in range(n_levels):
        g_tR[a, b] = sum(M[k, a] * Fraction(PRIMORIALS[k]) * M[k, b]
                         for k in range(n_levels))

# g_tR[0,:] and g_tR[:,0] have an implicit omega factor (column 0 of M)
# g_tR[0,0] has omega^2 factor
# Display with omega explicit

print("g_{(t,R)} = M^T W M  (omega factors shown explicitly):")
print()
for a in range(n_levels):
    row = []
    for b in range(n_levels):
        val = g_tR[a, b]
        # omega powers: column 0 contributes omega once per index
        n_omega = (1 if a == 0 else 0) + (1 if b == 0 else 0)
        prefix = ["", "ω·", "ω²·"][n_omega]
        row.append(f"{prefix}{val}")
    print(f"  [{', '.join(f'{s:>14s}' for s in row)}]")

# ── Verify tail-sum structure ──
print("\nTail sums s_m = sum_{k>=m} 1/P_k:")
tail_sums = []
for m in range(n_levels):
    s = sum(Fraction(1, PRIMORIALS[k]) for k in range(m, n_levels))
    tail_sums.append(s)
    print(f"  s_{m} = {s} = {float(s):.10f}")

# Define weights: w_0 = 1 (omega factored), w_{j+1} = P_j
weights = [Fraction(1)] + [Fraction(PRIMORIALS[j]) for j in range(n_levels - 1)]

print("\nWeights w_a: [omega] + P_j =", [f"ω" if a == 0 else str(weights[a]) for a in range(n_levels)])

# Verify: g[a,b] = w_a * w_b * s_{max(a',b')+1}
# where a' = a for a>0, and for a=0 (time), we use s_0
print("\nTail-sum structure verification:")
all_ok = True
for a in range(n_levels):
    for b in range(a, n_levels):
        # For (t,R) mixing: the tail sum index is max of the R-levels involved
        # g[0,0] = omega^2 * s_0
        # g[0,j+1] = omega * P_j * s_{j+1}
        # g[i+1,j+1] = P_i * P_j * s_{max(i,j)+1}  for i <= j
        if a == 0 and b == 0:
            expect = tail_sums[0]  # * omega^2
        elif a == 0:
            j = b - 1
            expect = weights[b] * tail_sums[j + 1]  # * omega
        else:
            i, j = a - 1, b - 1
            expect = weights[a] * weights[b] * tail_sums[max(i, j) + 1]
        
        actual = g_tR[a, b]
        if a == 0 and b == 0:
            ok = actual == expect
        elif a == 0:
            ok = actual == expect
        else:
            ok = actual == expect
        all_ok &= ok
        if not ok:
            print(f"  MISMATCH at ({a},{b}): expect {expect}, got {actual}")

print(f"  All entries match tail-sum formula: {all_ok}  ✓" if all_ok else "  MISMATCHES found!")

g_{(t,R)} = M^T W M  (omega factors shown explicitly):

  [    ω²·179/105,       ω·74/105,       ω·43/105,         ω·8/35,          ω·1/7]
  [      ω·74/105,         74/105,         43/105,           8/35,            1/7]
  [      ω·43/105,         43/105,         86/105,          16/35,            2/7]
  [        ω·8/35,           8/35,          16/35,          48/35,            6/7]
  [         ω·1/7,            1/7,            2/7,            6/7,           30/7]

Tail sums s_m = sum_{k>=m} 1/P_k:
  s_0 = 179/105 = 1.7047619048
  s_1 = 74/105 = 0.7047619048
  s_2 = 43/210 = 0.2047619048
  s_3 = 4/105 = 0.0380952381
  s_4 = 1/210 = 0.0047619048

Weights w_a: [omega] + P_j = ['ω', '1', '2', '6', '30']

Tail-sum structure verification:
  All entries match tail-sum formula: True  ✓


## §5 — Metric Determinant and Volume Form

The metric determinant gives the volume element of the solenoid in
$(t, R)$ coordinates:

$$\det(g) = \det(M)^2 \cdot \det(W) = \frac{\omega^2}{P_4^2} \cdot \prod_{k=0}^{4} P_k$$

In [9]:
# ── Metric determinant (exact) ──
# det(g) = det(M)^2 * det(W)
# det(M) = omega/P_4, so det(g) = omega^2 * prod(P_k) / P_4^2

prod_Pk = det_W  # = 75600
det_g_ratio = prod_Pk / Fraction(P4 * P4)  # det(g) / omega^2
print(f"det(g) / omega^2 = prod(P_k) / P_4^2 = {prod_Pk} / {P4**2} = {det_g_ratio}")
print(f"  = {det_g_ratio.numerator}/{det_g_ratio.denominator}")

# Verify: 12/7 = lambda(210)/p_4
lam210 = 12
assert det_g_ratio == Fraction(lam210, primes[-1])
print(f"  = lambda(P_4) / p_4 = {lam210}/{primes[-1]}  [verified]")
print()

# Numerical value
det_g = float(det_g_ratio) * OMEGA**2
print(f"det(g) = omega^2 * 12/7 = (2*pi)^2 * 12/7 = {det_g:.6f}")
v1 = 48 * np.pi**2 / 7
print(f"       = 48*pi^2/7 = {v1:.6f}")
print(f"       = phi(P_4) * pi^2 * 4 / p_4")
print()

# Volume form
sqrt_det_g = OMEGA * np.sqrt(float(det_g_ratio))
print(f"sqrt(det g) = omega * sqrt(lambda(P_4)/p_4) = {sqrt_det_g:.6f}")
v2 = 4 * np.pi * np.sqrt(3) / np.sqrt(7)
print(f"            = 4*pi*sqrt(3)/sqrt(7) = {v2:.6f}")
print()

# Direct verification of determinant from the matrix
g_float = np.array([[float(g_tR[i, j]) for j in range(n_levels)] for i in range(n_levels)])
g_actual = g_float.copy()
g_actual[0, :] *= OMEGA
g_actual[:, 0] *= OMEGA
det_numerical = np.linalg.det(g_actual)
print(f"Numerical det(g) = {det_numerical:.6f}  (exact: {det_g:.6f})")
print(f"Relative error: {abs(det_numerical - det_g) / det_g:.2e}")

# Eigenvalues of the full metric (with omega restored)
eigvals_g = np.sort(np.linalg.eigvalsh(g_actual))
print(f"\nMetric eigenvalues: {eigvals_g}")
all_pos = all(e > 0 for e in eigvals_g)
sig = "Riemannian +5" if all_pos else "INDEFINITE"
print(f"  All positive: {all_pos}  ({sig})")
print(f"  Trace: {sum(eigvals_g):.6f}")
print(f"  Product: {np.prod(eigvals_g):.6f}  (= det = {det_g:.6f})")

det(g) / omega^2 = prod(P_k) / P_4^2 = 75600 / 44100 = 12/7
  = 12/7
  = lambda(P_4) / p_4 = 12/7  [verified]

det(g) = omega^2 * 12/7 = (2*pi)^2 * 12/7 = 67.677287
       = 48*pi^2/7 = 67.677287
       = phi(P_4) * pi^2 * 4 / p_4

sqrt(det g) = omega * sqrt(lambda(P_4)/p_4) = 8.226621
            = 4*pi*sqrt(3)/sqrt(7) = 8.226621

Numerical det(g) = 67.677287  (exact: 67.677287)
Relative error: 8.40e-16

Metric eigenvalues: [ 0.27397663  0.60360781  1.33364538  4.52982392 67.74120104]
  All positive: True  (Riemannian +5)
  Trace: 74.482255
  Product: 67.677287  (= det = 67.677287)


### Identity #184: Metric Determinant

$$\det(g_{(t,R)}) = \omega^2 \cdot \frac{\lambda(P_4)}{p_4}
  = \frac{4\pi^2 \cdot 12}{7} = \frac{48\pi^2}{7}
  = \frac{\varphi(P_4) \cdot \pi^2 \cdot 4}{p_4}$$

The **volume form** of the solenoid in natural coordinates is
controlled by the group exponent $\lambda(210) = 12$ and the
outermost prime $p_4 = 7$:

$$\sqrt{\det g} = \omega \sqrt{\frac{\lambda(P_4)}{p_4}}
  = \frac{4\pi\sqrt{3}}{\sqrt{7}}$$

The metric eigenvalues are all positive (Riemannian signature),
with the largest eigenvalue ($\approx 67.7$) close to $\det(g)$ itself
— indicative of a dominant time direction.

## §6 — Stiffness in $(t, R)$ Coordinates: Time as Zero Mode

The covering constraint potential in $(t, R)$ coordinates is
$V = (\kappa/2)\,x^\top K_{(t,R)}\,x$ where $K_{(t,R)} = M^\top K M$.

Since $K v_0 = 0$ and $v_0$ maps to the time direction
$e_0 = (1, 0, 0, 0, 0)$ in $(t, R)$ coordinates,
the time row/column of $K_{(t,R)}$ must vanish:

$$K_{(t,R)}[0, :] = K_{(t,R)}[:, 0] = \mathbf{0}$$

This is the **Metric Separation Principle in Lagrangian form**:
time is elastically free, all stiffness resides in the $R$-directions.

In [10]:
# ── Stiffness in (t,R) coordinates: K_{tR} = M^T K M ──
K_tR = np.zeros((n_levels, n_levels), dtype=object)
for a in range(n_levels):
    for b in range(n_levels):
        K_tR[a, b] = sum(M[k, a] * K[k, l] * M[l, b]
                         for k in range(n_levels)
                         for l in range(n_levels))

print("K_{(t,R)} = M^T K M:")
print()
for a in range(n_levels):
    row = []
    for b in range(n_levels):
        val = K_tR[a, b]
        n_omega = (1 if a == 0 else 0) + (1 if b == 0 else 0)
        prefix = ["", "w*", "w2*"][n_omega]
        row.append(f"{prefix}{val}")
    print(f"  [{', '.join(f'{s:>12s}' for s in row)}]")

# Verify time row/column vanishes
time_row = [K_tR[0, b] for b in range(n_levels)]
time_col = [K_tR[a, 0] for a in range(n_levels)]
print(f"\nTime row:    {time_row}")
print(f"Time column: {time_col}")
all_zero = all(x == 0 for x in time_row + time_col)
print(f"All zero: {all_zero}  [time is elastically free]")

# The 4x4 R-block of K_{tR}
print("\nR-block of K_{(t,R)}:")
K_R = np.array([[K_tR[i+1, j+1] for j in range(n_levels-1)] for i in range(n_levels-1)])
for i in range(4):
    row = [f"{K_R[i,j]}" for j in range(4)]
    print(f"  [{', '.join(f'{s:>12s}' for s in row)}]")

# Eigenvalues of the R-block (should match omega^2 from W^{-1}K)
K_R_float = np.array([[float(K_R[i,j]) for j in range(4)] for i in range(4)])
g_R_float = np.array([[float(g_tR[i+1,j+1]) for j in range(4)] for i in range(4)])

# Generalized eigenvalue problem: K_R v = omega^2 g_R v
from scipy.linalg import eigh
omega_sq_tR, _ = eigh(K_R_float, g_R_float)
print(f"\nomega^2 from (t,R) coordinates: {np.sort(omega_sq_tR)}")
print(f"omega^2 from theta coordinates: {omega_sq}")
print(f"Match: {np.allclose(np.sort(omega_sq_tR), omega_sq)}")
print(f"  [Eigenvalues are coordinate-invariant]")

K_{(t,R)} = M^T K M:

  [        w2*0,          w*0,          w*0,          w*0,          w*0]
  [         w*0,            1,            0,            0,            0]
  [         w*0,            0,            1,            0,            0]
  [         w*0,            0,            0,            1,            0]
  [         w*0,            0,            0,            0,            1]

Time row:    [Fraction(0, 1), Fraction(0, 1), Fraction(0, 1), Fraction(0, 1), Fraction(0, 1)]
Time column: [Fraction(0, 1), Fraction(0, 1), Fraction(0, 1), Fraction(0, 1), Fraction(0, 1)]
All zero: True  [time is elastically free]

R-block of K_{(t,R)}:
  [           1,            0,            0,            0]
  [           0,            1,            0,            0]
  [           0,            0,            1,            0]
  [           0,            0,            0,            1]

omega^2 from (t,R) coordinates: [0.21849319 0.65540166 1.33002746 3.06274436]
omega^2 from theta coordinates: [0.22062139

### Identity #185: Unit Stiffness in Natural Coordinates

$$K_{(t,R)} = \mathrm{diag}(0, 1, 1, 1, 1)$$

In $(t, R)$ coordinates, the covering constraint potential reduces to:

$$V = \frac{\kappa}{2}\sum_{k=0}^{3} R_k^2$$

Each covering residual has **unit stiffness**. The entire geometric complexity 
of the solenoid coupling — the tridiagonal matrix $K$ with entries up to 
$p_4^2 = 49$ — is absorbed into the coordinate transformation.

This means the **eigenfrequencies $\omega_k^2$ arise entirely from the metric geometry** 
(the non-trivial $g_R$ block), not from the stiffness. The oscillation modes 
are determined by the shape of the space, not the strength of the springs.

The eigenvalue problem in $(t,R)$ coordinates requires the **Schur complement** 
of $g_{(t,R)}$ to eliminate the time-R coupling:

$$\tilde{g}_R = g_{RR} - \frac{1}{g_{tt}} g_{Rt} g_{tR}$$

The oscillation frequencies are $\omega^2_k = 1 / \lambda_k(\tilde{g}_R)$.

In [11]:
# ── Schur complement: eliminate time-R coupling ──
# g_tilde_R = g_RR - (1/g_tt) * g_tR^T * g_tR
# (Note: omega cancels — g_tt has omega^2, g_tR has omega, product has omega^2)

g_tt = g_tR[0, 0]  # = 179/105 (× omega^2)
g_tR_vec = np.array([g_tR[0, j] for j in range(1, n_levels)])  # (× omega)

# Schur complement (omega factors cancel in the ratio)
g_schur = np.zeros((4, 4), dtype=object)
for i in range(4):
    for j in range(4):
        g_schur[i, j] = g_tR[i + 1, j + 1] - g_tR_vec[i] * g_tR_vec[j] / g_tt

print("Schur complement g_tilde_R = g_RR - g_tR^T g_tR / g_tt:")
print()
for i in range(4):
    row = [f"{g_schur[i,j]}" for j in range(4)]
    print(f"  [{', '.join(f'{s:>14s}' for s in row)}]")

# eigenvalues of the Schur complement
g_schur_float = np.array([[float(g_schur[i, j]) for j in range(4)] for i in range(4)])
eigvals_schur = np.sort(np.linalg.eigvalsh(g_schur_float))
print(f"\nEigenvalues of Schur complement: {eigvals_schur}")

# omega^2 = 1/eigenvalues
omega_sq_schur = 1.0 / eigvals_schur[::-1]  # reverse so smallest omega first
print(f"omega^2 = 1/lambda(g_tilde):     {np.sort(omega_sq_schur)}")
print(f"omega^2 from theta-space:         {omega_sq}")
print(f"Match: {np.allclose(np.sort(omega_sq_schur), omega_sq)}")
print(f"  [Eigenvalues ARE coordinate-invariant via Schur complement]")

# ── What determines the Schur complement? ──
# det(g_tilde_R)
det_schur = np.linalg.det(g_schur_float)
print(f"\ndet(g_tilde_R) = {det_schur:.10f}")
# det(g_tilde_R) = det(g) / g_tt (Schur complement determinant identity)
print(f"det(g) / g_tt = {det_g / (float(g_tt) * OMEGA**2):.10f}")

# Exact: det(g)/g_tt = (omega^2 * 12/7) / (omega^2 * 179/105) = (12/7)*(105/179) = 12*15/179 = 180/179
det_schur_exact = Fraction(lam210, primes[-1]) / g_tt
print(f"Exact: {det_schur_exact} = {float(det_schur_exact):.10f}")
print(f"     = 180/179 = 1/Pi(omega^2) !")
assert det_schur_exact == Fraction(180, 179)

# Since omega^2 = 1/eigenvalues(g_tilde):
# prod(omega^2) = 1/prod(eigenvalues) = 1/det(g_tilde) = 179/180  ✓
print(f"\n1/det(g_tilde) = {Fraction(179, 180)} = Pi(omega^2)  [consistent]")

Schur complement g_tilde_R = g_RR - g_tR^T g_tR / g_tt:

  [        74/179,         43/179,         24/179,         15/179]
  [        43/179,        129/179,         72/179,         45/179]
  [        24/179,         72/179,        240/179,        150/179]
  [        15/179,         45/179,        150/179,        765/179]

Eigenvalues of Schur complement: [0.27434404 0.60503152 1.33657568 4.53265211]
omega^2 = 1/lambda(g_tilde):     [0.22062139 0.7481806  1.65280644 3.64505823]
omega^2 from theta-space:         [0.22062139 0.7481806  1.65280644 3.64505823]
Match: True
  [Eigenvalues ARE coordinate-invariant via Schur complement]

det(g_tilde_R) = 1.0055865922
det(g) / g_tt = 1.0055865922
Exact: 180/179 = 1.0055865922
     = 180/179 = 1/Pi(omega^2) !

1/det(g_tilde) = 179/180 = Pi(omega^2)  [consistent]


### Identity #186: Schur Complement Determinant

$$\det(\tilde{g}_R) = \frac{180}{179} = \frac{1}{\prod \omega_i^2}$$

The Schur complement — the "effective R-metric after eliminating time" — has 
determinant equal to the **inverse eigenvalue product**. This is the direct 
metric-theoretic expression of identity #42 ($\prod\omega^2 = 179/180$).

All entries of $\tilde{g}_R$ share **denominator 179** (from $g_{tt} = 179/105$). 
The numerator of the $(2,2)$ entry is $240 = \mathrm{Tr}(L) = c_1(E_4) = |\Phi(E_8)|$ —
the Cayley graph spectral trace (identity #59) appearing in the geometric 
sector metric. This is a concrete realization of the Metric Separation 
Principle: the flat spectral sector embeds into the curved geometric sector 
through the Schur complement.

## §7 — The Cascade Conformal Factor

The cascade ODE (NB79–81) gives $R_k(t) = R_k(0)\,e^{-\kappa t} + \text{steady state}$
with $\kappa = 1/\sqrt{P_4}$. The exponential decay defines a natural 
**conformal factor** on the $(t,R)$ metric:

$$ds^2(t) = e^{-2\kappa t}\, ds^2_{\text{spatial}}$$

This is a de Sitter-like evolution with "Hubble parameter" $H = \kappa = \rho$.
The same $1/\sqrt{210}$ that governs fermion mass ratios also governs the 
rate of solenoid equilibration.

In [12]:
# ── Cascade conformal factor ──
from solenoid_system import SolenoidSystem

kappa = RHO  # 1/sqrt(210)

print("CASCADE CONFORMAL FACTOR")
print("=" * 65)
print(f"  kappa = rho = 1/sqrt(P_4) = 1/sqrt({P4}) = {kappa:.6f}")
print(f"  Decay timescale: tau = 1/kappa = sqrt({P4}) = {1/kappa:.4f}")
print()

# At the physical crossings, the decay factor is exp(-kappa*(ci+1))
physical_crossings = [11, 31, 61, 191]
crossing_names = ["QUARK_g1", "LEPTON_g1", "LEPTON_g2", "QUARK_g2"]

print(f"  {'Crossing':<12s}  {'ci':>4s}  {'exp(-kappa*ci)':>16s}  {'Conformal factor':>16s}")
print(f"  {'-'*55}")
for name, ci in zip(crossing_names, physical_crossings):
    decay = np.exp(-kappa * (ci + 1))
    conformal = decay**2
    print(f"  {name:<12s}  {ci:4d}  {decay:16.6f}  {conformal:16.6f}")

# The "age" at which conformal factor = Omega_Lambda
omega_lambda = Fraction(8, 35)  # NB37 prediction
t_lambda = -np.log(float(omega_lambda)) / (2 * kappa)
print(f"\n  Omega_Lambda = {omega_lambda} = {float(omega_lambda):.6f}")
print(f"  exp(-2*kappa*t) = Omega_Lambda at t = {t_lambda:.2f}")
print(f"  (in units of base-circle periods)")
print()

# The connection: rho = kappa = epsilon
print("THE TRIPLE ROLE OF 1/sqrt(P_4):")
print(f"  rho   = VEV profile parameter (mass ratios)")
print(f"  kappa = linear restoring strength (cascade ODE)")
print(f"  epsilon = perturbation strength (solenoid deviation)")
print(f"  H_sol = cascade Hubble parameter (conformal decay)")
print(f"  All = 1/sqrt({P4}) = {kappa:.6f}")
print()

# ── Metric summary table ──
print("METRIC SUMMARY")
print("=" * 65)
print(f"  Configuration metric:  W = diag(1, 2, 6, 30, 210)")
print(f"  det(W) = {int(det_W)} = 2^4 * 3^3 * 5^2 * 7")
print(f"  Zero-mode inertia:     g_1D = 179/105")
print(f"  Jacobian det(M):       omega / P_4")
print(f"  Metric det(g):         omega^2 * 12/7 = 48*pi^2/7")
print(f"  Stiffness K_(t,R):     diag(0, 1, 1, 1, 1)")
print(f"  Schur complement det:  180/179 = 1/Pi(omega^2)")
print(f"  Conformal rate:        kappa = 1/sqrt(210)")

CASCADE CONFORMAL FACTOR
  kappa = rho = 1/sqrt(P_4) = 1/sqrt(210) = 0.069007
  Decay timescale: tau = 1/kappa = sqrt(210) = 14.4914

  Crossing        ci    exp(-kappa*ci)  Conformal factor
  -------------------------------------------------------
  QUARK_g1        11          0.436888          0.190871
  LEPTON_g1       31          0.109897          0.012077
  LEPTON_g2       61          0.013865          0.000192
  QUARK_g2       191          0.000002          0.000000

  Omega_Lambda = 8/35 = 0.228571
  exp(-2*kappa*t) = Omega_Lambda at t = 10.69
  (in units of base-circle periods)

THE TRIPLE ROLE OF 1/sqrt(P_4):
  rho   = VEV profile parameter (mass ratios)
  kappa = linear restoring strength (cascade ODE)
  epsilon = perturbation strength (solenoid deviation)
  H_sol = cascade Hubble parameter (conformal decay)
  All = 1/sqrt(210) = 0.069007

METRIC SUMMARY
  Configuration metric:  W = diag(1, 2, 6, 30, 210)
  det(W) = 75600 = 2^4 * 3^3 * 5^2 * 7
  Zero-mode inertia:     g_1D = 

## §8 — Scorecard

NB82 establishes the **solenoid metric**: the Riemannian metric on the configuration space of the (2,3,5,7)-solenoid in natural (t, R) coordinates. Six structural identities:

| # | Identity | Solenoid Value | Status |
|---|----------|---------------|--------|
| 181 | Zero-mode inertia g₁D = Σ 1/Pₖ | 179/105 (179 prime) | PASS |
| 182 | Product bridge: g₁D · p₄/λ(P₄) = Πω² | 179/180 | PASS |
| 183 | Jacobian determinant det(M) | ω/P₄ = 2π/210 | PASS |
| 184 | Metric determinant det(g) | ω² · λ(P₄)/p₄ = 48π²/7 | PASS |
| 185 | Stiffness in (t,R): K_{(t,R)} | diag(0, 1, 1, 1, 1) | STRUCTURAL |
| 186 | Schur complement det(g̃_R) | 180/179 = 1/Πω² | PASS |

**Key structural observation**: The Schur complement g̃_R has all entries with denominator 179. The numerator of the (2,2) entry is **240 = Tr(L) = c₁(E₄)** — the spectral trace of the Cayley graph embeds in the geometric metric through the Schur complement.

**Cascade conformal factor**: κ = 1/√210 serves simultaneously as coupling constant (mass ratios), restoring strength (cascade ODE), and conformal decay rate (metric evolution). The full H₀ derivation requires cosmological chain beyond this notebook's scope.

In [13]:
# ── Scorecard ──
print("NB82 SCORECARD: The Solenoid Metric")
print("=" * 65)

identities = [
    (181, "Zero-mode inertia g_1D",      "179/105",            "179/105",     "PASS"),
    (182, "Product bridge g_1D*p4/lam",   "179/180",            "179/180",     "PASS"),
    (183, "Jacobian det(M)",              "omega/P4",           "2*pi/210",    "PASS"),
    (184, "Metric det(g)",                "omega^2*lam/p4",     "48*pi^2/7",   "PASS"),
    (185, "Unit stiffness K_(t,R)",       "diag(0,1,1,1,1)",   "exact",       "STRUCTURAL"),
    (186, "Schur complement det",         "180/179",            "1/Pi(w^2)",   "PASS"),
]

print(f"  {'#':<4s} {'Identity':<28s} {'Value':<18s} {'Match':<14s} {'Verdict'}")
print(f"  {'-'*80}")
for num, name, val, match, verdict in identities:
    print(f"  {num:<4d} {name:<28s} {val:<18s} {match:<14s} {verdict}")

n_pass = sum(1 for _, _, _, _, v in identities if v == 'PASS')
n_struct = sum(1 for _, _, _, _, v in identities if v == 'STRUCTURAL')

print(f"\n  NB82: {n_pass} PASS + {n_struct} STRUCTURAL = {n_pass + n_struct} new identities")
print(f"  Running total: 171 + {n_pass + n_struct} = {171 + n_pass + n_struct} structural identities")
print(f"  Free parameters: 0")
print(f"  Dimensional anchor: M_Z (1)")

NB82 SCORECARD: The Solenoid Metric
  #    Identity                     Value              Match          Verdict
  --------------------------------------------------------------------------------
  181  Zero-mode inertia g_1D       179/105            179/105        PASS
  182  Product bridge g_1D*p4/lam   179/180            179/180        PASS
  183  Jacobian det(M)              omega/P4           2*pi/210       PASS
  184  Metric det(g)                omega^2*lam/p4     48*pi^2/7      PASS
  185  Unit stiffness K_(t,R)       diag(0,1,1,1,1)    exact          STRUCTURAL
  186  Schur complement det         180/179            1/Pi(w^2)      PASS

  NB82: 5 PASS + 1 STRUCTURAL = 6 new identities
  Running total: 171 + 6 = 177 structural identities
  Free parameters: 0
  Dimensional anchor: M_Z (1)
